# Job Shop Scheduling using Genetic Algorithm
**AI Spring 2026 — Project 2, Part B (R3, R4, R5)**

In [4]:
import random
import itertools

## Core Functions

In [5]:
def compute_makespan(chromosome, jobs, num_machines):
    
    # Simulating the schedule for a given job ordering and return the makespan.
    # Operations are assigned to machines in round-robin (op1->M1, op2->M2, etc).
    machine_free_at = [0] * num_machines

    for job_id in chromosome:
        job_end = 0  # when this job's previous operation finished

        for op_num, op_time in enumerate(jobs[job_id]):
            m = op_num % num_machines
            start = max(machine_free_at[m], job_end)
            end = start + op_time
            machine_free_at[m] = end
            job_end = end

    return max(machine_free_at)

In [6]:
def generate_population(num_jobs, pop_size=None):
    
    # For small job counts, generate all permutations.
    # For larger ones, generate pop_size number of random permutations.
    if pop_size is None and num_jobs <= 8:
        return list(itertools.permutations(range(num_jobs)))

    if pop_size is None:
        pop_size = 120

    population = set()
    while len(population) < pop_size:
        perm = tuple(random.sample(range(num_jobs), num_jobs))
        population.add(perm)
    return list(population)

In [7]:
def select_chromosomes(population, jobs, num_machines, num_select=10):
    # chromosomes with lower makespan get picked more often 
    # (probability proportional to 1/makespan).
    fitnesses = [compute_makespan(c, jobs, num_machines) for c in population]

    # lower makespan = better, so we use inverse
    inv = [1.0 / f for f in fitnesses]
    total = sum(inv)
    probs = [x / total for x in inv]

    selected = random.choices(population, weights=probs, k=num_select)
    return selected

In [8]:
def crossover(p1, p2):
    
    # Two-cut interval crossover.
    # Keeping the middle section from one parent, fill the rest from the other.
    n = len(p1)
    cut1, cut2 = sorted(random.sample(range(1, n), 2))

    def make_child(parent_a, parent_b):
        middle = list(parent_a[cut1:cut2])
        middle_set = set(middle)
        # take everything from parent_b that isn't already in the middle
        rest = [g for g in parent_b if g not in middle_set]
        # stitch together: rest[:cut1] + middle + rest[cut1:]
        return tuple(rest[:cut1] + middle + rest[cut1:])

    child1 = make_child(p1, p2)
    child2 = make_child(p2, p1)
    return child1, child2

In [9]:
def crossover_all_pairs(selected):
    # Applying crossover to every unique pair from the selected chromosomes.
    children = []
    for i in range(len(selected)):
        for j in range(i + 1, len(selected)):
            c1, c2 = crossover(selected[i], selected[j])
            children.append(c1)
            children.append(c2)
    return children

In [10]:
def mutate(chromosome, prob=0.05):
    # With given probability, swap two random positions in the chromosome.
    if random.random() < prob:
        lst = list(chromosome)
        i, j = random.sample(range(len(lst)), 2)
        lst[i], lst[j] = lst[j], lst[i]
        return tuple(lst)
    return chromosome

## Main GA Loop

In [11]:
def run_ga(jobs, num_machines, generations=100, pop_size=None, select_size=10, mut_prob=0.05):
    """
    Full GA pipeline:
    1. Generate initial population
    2. For each generation: select -> crossover all pairs -> mutate -> add to population
    3. Return best chromosome and its makespan
    """
    num_jobs = len(jobs)
    population = generate_population(num_jobs, pop_size)
    print(f"Initial population size: {len(population)}")

    random_selected_makespan = None

    for gen in range(generations):
        # select 10 chromosomes
        selected = select_chromosomes(population, jobs, num_machines, select_size)

        # save a random one from the first generation's selection for comparison
        if gen == 0:
            rand_chrom = random.choice(selected)
            random_selected_makespan = compute_makespan(rand_chrom, jobs, num_machines)

        # crossover all 45 pairs
        offspring = crossover_all_pairs(selected)

        # mutate each offspring
        offspring = [mutate(c, mut_prob) for c in offspring]

        # add new chromosomes to the population (deduplicate)
        population = list(set(population + offspring))

        # print progress
        # if (gen + 1) % 25 == 0:
        #     best_so_far = min(compute_makespan(c, jobs, num_machines) for c in population)
        #     print(f"Gen {gen+1}: best makespan = {best_so_far}, pop size = {len(population)}")

    # find the overall best
    best_chrom = min(population, key=lambda c: compute_makespan(c, jobs, num_machines))
    best_makespan = compute_makespan(best_chrom, jobs, num_machines)

    return best_chrom, best_makespan, rand_chrom, random_selected_makespan

## Helper Functions

In [12]:
def generate_random_jobs(num_jobs, num_ops, low=5, high=50):
    # Generating random jobs with operation times in [low, high].
    return [tuple(random.randint(low, high) for _ in range(num_ops))
            for _ in range(num_jobs)]


def print_jobs(jobs):
    # Print the job table nicely.
    num_ops = len(jobs[0])
    header = "Job#  " + "  ".join([f"Op{i+1:>2}" for i in range(num_ops)])
    print(header)
    print("-" * len(header))
    for i, job in enumerate(jobs):
        ops_str = "  ".join([f"{t:>4}" for t in job])
        print(f"  {i+1}   {ops_str}")


def print_chromosome(chromosome):
    display = tuple(i+1 for i in chromosome)
    print(f"Chromosome: {display}")

In [13]:
# Table 1 from Project 2 Discussion Document
# 5 jobs, each with 2 operations (Op1 time, Op2 time)
r3_jobs = [
    (3, 6),   # Job 1
    (10, 1),  # Job 2
    (3, 2),   # Job 3
    (2, 4),   # Job 4
    (8, 8),   # Job 5
]

In [14]:
# quick check — the tutorial says schedule (4,1,5,3,2) gives makespan 27
test = compute_makespan((3, 0, 4, 2, 1), r3_jobs, 2)  # 0-indexed version of (4,1,5,3,2)
print(f"Tutorial schedule (4,1,5,3,2) makespan: {test}  (optimal 27)")

Tutorial schedule (4,1,5,3,2) makespan: 27  (optimal 27)


In [15]:
print("R3: 5 Jobs, 2 Operations each, 2 Machines")
print_jobs(r3_jobs)

random.seed(42)
r3_best, r3_makespan, _, _ = run_ga(r3_jobs, num_machines=2)

print("\nGA Best Schedule: ") 
print_chromosome(r3_best)
print(f"GA Best makespan: {r3_makespan}")
print(f"Optimal makespan (from document): 30")
print(f"Comparing with Optimal: GA found optimal: {r3_makespan == 30}")


R3: 5 Jobs, 2 Operations each, 2 Machines
Job#  Op 1  Op 2
----------------
  1      3     6
  2     10     1
  3      3     2
  4      2     4
  5      8     8
Initial population size: 120

GA Best Schedule: 
Chromosome: (1, 5, 4, 3, 2)
GA Best makespan: 27
Optimal makespan (from document): 30
Comparing with Optimal: GA found optimal: False


---
# R4: 10 Random Jobs, 3 Operations, 5 Machines

In [16]:
random.seed(123)
r4_jobs = generate_random_jobs(10, 3)

print("="*55)
print("R4: 10 Jobs, 3 Operations each, 5 Machines")
print("="*55)
print()
print_jobs(r4_jobs)
print()


r4_best, r4_makespan, r4_rand, r4_rand_makespan = run_ga(r4_jobs, num_machines=5, pop_size=120)

print(f"\n--- R4 Answers ---")

print("\nRandomly selected schedule from the initial population: ")
print_chromosome(r4_rand)
print(f"Random selected chromosome makespan:  {r4_rand_makespan}")

print("Best schedule: ")
print_chromosome(r4_best)
print(f"Best makespan (highest fitness):      {r4_makespan}")



diff = r4_rand_makespan - r4_makespan
print(f"\nImprovement: {diff} units ({diff/r4_rand_makespan*100:.1f}% reduction)")

R4: 10 Jobs, 3 Operations each, 5 Machines

Job#  Op 1  Op 2  Op 3
----------------------
  1      8    22    10
  2     31    22    11
  3      7    29    39
  4     40    26    26
  5      8    15    13
  6     26    40    26
  7     49    20    15
  8      5    32    10
  9     43    29     9
  10      5    25    33

Initial population size: 120

--- R4 Answers ---

Randomly selected schedule from the initial population: 
Chromosome: (6, 8, 10, 9, 4, 3, 5, 1, 2, 7)
Random selected chromosome makespan:  301
Best schedule: 
Chromosome: (8, 1, 10, 7, 3, 6, 5, 4, 2, 9)
Best makespan (highest fitness):      274

Improvement: 27 units (9.0% reduction)


---
# R5: 10 Random Jobs, 5 Operations, 3 Machines

In [17]:
random.seed(456)
r5_jobs = generate_random_jobs(10, 5)

print("="*55)
print("R5: 10 Jobs, 5 Operations each, 3 Machines")
print("="*55)
print()
print_jobs(r5_jobs)
print()

r5_best, r5_makespan, r5_rand, r5_rand_makespan = run_ga(r5_jobs, num_machines=3, pop_size=120)

print(f"\n--- R5 Answers ---")

print("\nRandomly selected schedule from the initial population: ")
print_chromosome(r5_rand)
print(f"Random selected chromosome makespan:  {r5_rand_makespan}")

print("Best schedule: ")
print_chromosome(r5_best)
print(f"Best makespan (highest fitness):      {r5_makespan}")

diff = r5_rand_makespan - r5_makespan
print(f"\nImprovement: {diff} units ({diff/r5_rand_makespan*100:.1f}% reduction)")
print()

R5: 10 Jobs, 5 Operations each, 3 Machines

Job#  Op 1  Op 2  Op 3  Op 4  Op 5
----------------------------------
  1     33    34    32    29    46
  2      7    44    38    12    25
  3     36    12    10    35    49
  4     34     7    21    14    13
  5     17    16    29    38    44
  6     39    50    46    42    40
  7     49    48    35    26     9
  8     41    39     7    28    49
  9     24    46    31    17    27
  10     13    19    33    13    48

Initial population size: 120

--- R5 Answers ---

Randomly selected schedule from the initial population: 
Chromosome: (3, 10, 8, 7, 9, 1, 2, 4, 6, 5)
Random selected chromosome makespan:  1293
Best schedule: 
Chromosome: (2, 9, 5, 3, 8, 7, 10, 6, 1, 4)
Best makespan (highest fitness):      1212

Improvement: 81 units (6.3% reduction)

